# 韩国造船业分析

本Notebook用于分析韩国造船业反内卷与转型升级相关内容。

## 1. 环境配置与依赖导入

In [ ]:
# 导入标准库
import os
import sys
import datetime

# 导入数据处理库
import pandas as pd
import numpy as np

# 导入可视化库
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 导入数据库库
import sqlalchemy
from sqlalchemy import create_engine

# 导入配置
from config import DATABASE_URL, PROJECT_NAME

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print(f"项目: {PROJECT_NAME}")
print(f"当前时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. 资料加载与整理

In [ ]:
# 列出当前目录下的所有文件
files = [f for f in os.listdir('.') if f.endswith('.md')]
print("当前目录下的Markdown文件:")
for f in files:
    print(f"  - {f}")

In [ ]:
# 读取文章草稿
article_files = ['文章草稿.md', '文章思路.md', '我的写作风格参考.md', '业务参考.md']
content = {}

for filename in article_files:
    if os.path.exists(filename):
        with open(filename, 'r', encoding='utf-8') as f:
            content[filename] = f.read()
            print(f"已加载: {filename} ({len(content[filename])} 字符)")

## 3. 内容分析与整理

In [ ]:
# 显示文章主要内容
if '反内卷专题：韩国造船业反内卷与转型升级的启示.md' in content:
    main_article = content['反内卷专题：韩国造船业反内卷与转型升级的启示.md']
    print("主要内容摘要:")
    print("="*60)
    # 显示前2000字符
    print(main_article[:2000])
    print("\n...")

In [ ]:
# 分析文章结构
def analyze_article_structure(text):
    """分析文章结构，提取标题"""
    lines = text.split('\n')
    headers = [line for line in lines if line.startswith('#')]
    return headers

for filename, text in content.items():
    print(f"\n{filename} 结构:")
    headers = analyze_article_structure(text)
    for h in headers[:10]:  # 显示前10个标题
        print(f"  {h}")

## 4. 关键词分析

In [ ]:
# 关键词统计
from collections import Counter
import jieba

def extract_keywords(text, top_n=20):
    """提取关键词"""
    words = jieba.cut(text)
    # 过滤停用词
    stop_words = set(['的', '是', '在', '和', '了', '与', '等', '也', '对', '这', '为', '到', '中', '将', '被', '由', '不'])
    filtered_words = [w for w in words if len(w) > 1 and w not in stop_words]
    word_count = Counter(filtered_words)
    return word_count.most_common(top_n)

if content:
    all_text = ' '.join(content.values())
    keywords = extract_keywords(all_text)
    print("\n文章关键词统计:")
    print("="*40)
    for word, count in keywords:
        print(f"  {word}: {count}")

## 5. 可视化分析

In [ ]:
# 关键词可视化
if content:
    keywords_for_plot = keywords[:15]
    words = [k[0] for k in keywords_for_plot]
    counts = [k[1] for k in keywords_for_plot]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(words, counts, color='steelblue')
    ax.set_xlabel('出现次数')
    ax.set_title('韩国造船业分析 - 关键词统计')
    ax.invert_yaxis()
    
    # 添加数值标签
    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
                str(count), va='center', fontsize=10)
    
    plt.tight_layout()
    os.makedirs('output', exist_ok=True)
    plt.savefig('output/keyword_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. 数据库存储

In [ ]:
# 将分析结果存储到数据库
def get_db_connection():
    """获取数据库连接"""
    try:
        engine = create_engine(DATABASE_URL, poolclass=sqlalchemy.pool.NullPool)
        conn = engine.connect()
        print("数据库连接成功")
        return engine, conn
    except Exception as e:
        print(f"数据库连接失败: {e}")
        return None, None

engine, conn = get_db_connection()

In [ ]:
# 存储关键词统计结果
if conn is not None and keywords:
    keyword_df = pd.DataFrame(keywords, columns=['keyword', 'count'])
    keyword_df['project'] = PROJECT_NAME
    keyword_df['created_at'] = datetime.datetime.now()
    
    try:
        keyword_df.to_sql('article_keywords', con=conn, if_exists='append', index=False)
        print(f"关键词数据已存入数据库，共 {len(keyword_df)} 条记录")
    except Exception as e:
        print(f"存储失败: {e}")

## 7. 结果输出与总结

In [ ]:
# 生成分析报告
print("\n" + "="*60)
print(f"项目: {PROJECT_NAME} - 分析报告")
print("="*60)

print(f"\n已加载文档数: {len(content)}")
for filename, text in content.items():
    print(f"  - {filename}: {len(text)} 字符")

print(f"\n提取关键词数: {len(keywords)}")
print("\n核心主题:")
for word, count in keywords[:5]:
    print(f"  - {word}: {count} 次")

print("\n" + "="*60)
print("分析完成！")
print("="*60)

In [ ]:
# 关闭数据库连接
if conn is not None:
    conn.close()
    print("数据库连接已关闭")